# VideoTool cloud GPU whisper — Kaggle runner (PRIMARY)

Whisper-only GPU pre-compute on a Kaggle T4. Pulls **only** the narration voice and
`*_vi.txt` script from a Google Drive folder via rclone, runs `large-v3` on GPU, and
pushes `captions.srt` (+ `chapters.json`) back to the same Drive folder.

Setup once: enable the **GPU T4** accelerator + **Internet** in the notebook settings, and
add a Kaggle Secret `RCLONE_CONF` holding the full `rclone.conf` text. See
`docs/cloud-gpu-whisper-setup.md`.


## 1. rclone from Kaggle Secret


In [ ]:
import os, subprocess
from pathlib import Path
from kaggle_secrets import UserSecretsClient

conf = UserSecretsClient().get_secret('RCLONE_CONF')
cfg = Path.home()/'.config'/'rclone'/'rclone.conf'
cfg.parent.mkdir(parents=True, exist_ok=True)
cfg.write_text(conf)
if not subprocess.run(['which','rclone']).returncode == 0:
    subprocess.run('curl -fsSL https://rclone.org/install.sh | sudo bash', shell=True, check=True)
subprocess.run(['rclone','listremotes'], check=True)  # sanity: should print 'gdrive:'

## 2. Fetch cloud core + install videotool

`videotool_cloud.py` lives in `gdrive:_VIDEOTOOL_SHARED/`. Default install is the public
git repo; pass a Drive-staged wheel path to `setup()` only if the repo is private.


In [ ]:
subprocess.run(['rclone','copy','gdrive:_VIDEOTOOL_SHARED/videotool_cloud.py','.'], check=True)
import videotool_cloud as vc
vc.setup()  # repo_ref defaults to git+https://github.com/pnd4189/video-tool@main
import torch; assert torch.cuda.is_available(), 'No GPU — enable the T4 accelerator.'

## 3. Pull the job (voice + script only)

Set `JOB_REMOTE` to the Drive folder. The include-filter transfers only audio + `*_vi.txt`
(+ any job.yaml / `_creative/` seed) — never Image/Video/Parallax/Music.


In [ ]:
JOB_REMOTE = 'gdrive:1. YOUTUBE AUDIO/.../CHAP N'  # <-- EDIT THIS
JOB = '/kaggle/working/job'
subprocess.run([
    'rclone','copy', JOB_REMOTE, JOB,
    '--include','*.mp3','--include','*.wav','--include','*.m4a',
    '--include','*_vi.txt','--include','job.yaml','--include','_creative/**',
    '--progress',
], check=True)
print(os.listdir(JOB))

## 4. Run whisper on GPU (`large-v3`, float16)


In [ ]:
WHISPER_MODEL = 'large-v3'
caps, chaps = vc.run_whisper(JOB, model=WHISPER_MODEL, device='cuda', compute_type='float16')
print('captions:', caps)
print('chapters:', chaps)

## 5. Push outputs back to Drive


In [ ]:
subprocess.run([
    'rclone','copy', JOB+'/outputs', JOB_REMOTE+'/outputs',
    '--include','captions.srt','--include','chapters.json','--progress',
], check=True)
subprocess.run(['rclone','lsf', JOB_REMOTE+'/outputs'])